In [101]:
import pandas as pd
import math
import numpy as np

In [102]:
data = pd.read_csv("tennis.csv")
data.columns
data.head()

,outlook,temp,humidity,windy,play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes


In [103]:
features = [feat for feat in data.columns]
features.remove("play")

features

['outlook', 'temp', 'humidity', 'windy']

In [104]:
class Node:
    def __init__(self):
        self.children = []
        self.value = ""
        self.isLeaf = False
        self.pred = ""

In [105]:
def entropy(examples):
    pos = 0
    neg = 0

    for _, row in examples.iterrows():
        if row["play"] == "yes":
            pos += 1
        else:
            neg += 1

    if pos == 0.0 or neg == 0.0:
        return 0.0
    else:
        p = pos / (pos + neg)
        n = neg / (pos + neg)
        return -(p * math.log(p, 2) + n * math.log(n, 2))

In [106]:
def info_gain(examples, attr):
    uniq = np.unique(examples[attr])
    gain = entropy(examples)

    for u in uniq:
        subdata = examples[examples[attr] == u]
        sub_e = entropy(subdata)
        gain -= (float(len(subdata)) / float(len(examples))) * sub_e

    return gain

In [107]:
def ID3(examples, attrs):
    root = Node()

    max_gain = 0
    max_feat = ""

    for feature in attrs:
        gain = info_gain(examples, feature)
        if gain > max_gain:
            max_gain = gain
            max_feat = feature

    root.value = max_feat
    uniq = np.unique(examples[max_feat])

    for u in uniq:
        subdata = examples[examples[max_feat] == u]

        if entropy(subdata) == 0.0:
            newNode = Node()
            newNode.isLeaf = True
            newNode.value = u
            newNode.pred = np.unique(subdata["play"])
            root.children.append(newNode)

        else:
            dummyNode = Node()
            dummyNode.value = u

            new_attrs = attrs.copy()
            new_attrs.remove(max_feat)

            child = ID3(subdata, new_attrs)
            dummyNode.children.append(child)

            root.children.append(dummyNode)

    return root

In [108]:
def printTree(root: Node, depth=0):
    for i in range(depth):
        print("\t", end="")

    print(root.value, end="")

    if root.isLeaf:
        print("->", root.pred)

    print()

    for child in root.children:
        printTree(child, depth + 1)



In [109]:
root = ID3(data,features)
printTree(root)


outlook
	overcast-> ['yes']

	rainy
		windy
			False-> ['yes']

			True-> ['no']

	sunny
		humidity
			high-> ['no']

			normal-> ['yes']

